# MedSAM2: segment a 3D CT by prompting a single slice

SAM 2 tracks objects across video frames with memory attention. A CT volume has the
same structure — consecutive slices barely differ — so one box on one slice can be
propagated through the whole stack.

This notebook runs the method on two cases and then **tests three claims** about how it
behaves, rather than asserting them.

**Runtime → Change runtime type → T4 GPU** before running anything.

- MedSAM2: https://github.com/bowang-lab/MedSAM2 · weights research/education only
- Paper: https://arxiv.org/abs/2504.03600

## 1 · Setup

In [ ]:
import torch

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then rerun."
print(torch.cuda.get_device_name(0))

In [ ]:
%%capture
!pip install -q git+https://github.com/rekalantar/medsam2-3d-ct.git
!pip install -q SimpleITK imageio huggingface_hub
!git clone -q https://github.com/bowang-lab/MedSAM2.git /content/MedSAM2
%cd /content/MedSAM2
!pip install -q -e ".[dev]"
!bash download.sh

In [ ]:
import os

CKPT = "/content/MedSAM2/checkpoints/MedSAM2_latest.pt"
assert os.path.exists(CKPT), "download.sh did not produce MedSAM2_latest.pt"
print(f"checkpoint {os.path.getsize(CKPT) / 1e6:.0f} MB")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

from medsam2_ct import (build_predictor, largest_component, load_volume,
                        save_gif, segment_volume, window_hu)

DATASET = "wanglab/CT_DeepLesion-MedSAM2"
predictor = build_predictor(config="configs/sam2.1_hiera_t512.yaml", checkpoint=CKPT)
print("ready")

## 2 · One function, any case

Fetch a case, window it, place a box on the largest lesion cross-section, propagate,
and score. The prompt comes from the ground-truth label — standard practice for
evaluating promptable models, since it simulates a perfect user prompt and isolates
propagation quality from whether a human drew a good box.

In [ ]:
def dice(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    total = a.sum() + b.sum()
    return 1.0 if total == 0 else 2 * (a & b).sum() / total


def load_case(case, margin=5):
    image = hf_hub_download(DATASET, filename=f"images/{case}_0000.nii.gz",
                            repo_type="dataset")
    label = hf_hub_download(DATASET, filename=f"labels/{case}.nii.gz",
                            repo_type="dataset")
    volume_hu, spacing = load_volume(image)
    truth = load_volume(label)[0] > 0

    key = int(truth.sum(axis=(1, 2)).argmax())
    ys, xs = np.where(truth[key])
    box = [int(xs.min()) - margin, int(ys.min()) - margin,
           int(xs.max()) + margin, int(ys.max()) + margin]

    return dict(name=case, hu=volume_hu, volume=window_hu(volume_hu, 400, 40),
                truth=truth, spacing=spacing, key=key, box=box,
                span=int(truth.any(axis=(1, 2)).sum()))


def run_case(case_dict):
    c = case_dict
    masks = largest_component(
        segment_volume(predictor, c["volume"], c["box"], c["key"]))
    c["masks"] = masks
    c["dice"] = dice(c["truth"], masks)
    c["dice_key"] = dice(c["truth"][c["key"]], masks[c["key"]])
    return c

## 3 · Two cases: a compact lesion and a long one

Propagation distance is the variable that matters. A lesion spanning seven slices only
tests three slices of travel; one spanning forty is a real test.

In [ ]:
SMALL = "000009_03_01_036-048"   # compact
LARGE = "000002_02_01_044-083"   # spans many more slices

cases = [run_case(load_case(c)) for c in (SMALL, LARGE)]

print(f"{'case':<24}{'slices':>7}{'HU range':>18}{'Dice':>8}{'key':>8}")
for c in cases:
    hu = f"{c['hu'].min():.0f} to {c['hu'].max():.0f}"
    print(f"{c['name']:<24}{c['span']:>7}{hu:>18}{c['dice']:>8.3f}{c['dice_key']:>8.3f}")

The **HU range** column matters more than it looks. DeepLesion volumes are sometimes
distributed already clipped to a soft-tissue range. If the span is around 400 or less,
the volume is pre-windowed and claim 1 below cannot be tested on it.

## 4 · Where the error lives

Per-slice Dice against distance from the prompted slice.

In [ ]:
fig, axes = plt.subplots(1, len(cases), figsize=(6 * len(cases), 3.6), squeeze=False)
for ax, c in zip(axes[0], cases):
    z = c["truth"].any(axis=(1, 2)).nonzero()[0]
    per_slice = [dice(c["truth"][i], c["masks"][i]) for i in z]
    ax.plot(z - c["key"], per_slice, "o-", color="#3FC1C9", linewidth=2)
    ax.axvline(0, color="#FF6B5B", linestyle="--", label="prompted slice")
    ax.set_title(f"{c['span']} slices — volume Dice {c['dice']:.3f}", fontsize=11)
    ax.set_xlabel("slices from prompt"); ax.set_ylabel("Dice")
    ax.set_ylim(0, 1.05); ax.legend()
plt.tight_layout()
plt.savefig("decay.png", dpi=150, bbox_inches="tight")

## 5 · The payoff figure

In [ ]:
c = cases[0]
z = (c["masks"] | c["truth"]).any(axis=(1, 2)).nonzero()[0]
lo, hi = max(0, z.min() - 2), min(len(c["volume"]), z.max() + 3)

save_gif(c["volume"][lo:hi], c["masks"][lo:hi], "propagation.gif", fps=4)
print(f"{os.path.getsize('propagation.gif') / 1e6:.2f} MB, {hi - lo} frames")

In [ ]:
import IPython.display

IPython.display.Image("propagation.gif")

## 6 · Three claims

Each of these is something the method is commonly said to require. Two hold. One
does not — which is worth more than if all three had.

### Claim 1 — windowing changes the result

CT spans thousands of Hounsfield units; the encoder takes 8-bit. The usual advice is
that naive full-range scaling leaves soft tissue with no dynamic range. Does it
actually change the segmentation?

In [ ]:
c = cases[0]
naive = ((c["hu"] - c["hu"].min()) /
         (c["hu"].max() - c["hu"].min()) * 255).astype(np.uint8)

masks_naive = largest_component(
    segment_volume(predictor, naive, c["box"], c["key"]))

print(f"HU range        {c['hu'].min():.0f} to {c['hu'].max():.0f}")
print(f"windowed  Dice  {c['dice']:.3f}   {c['masks'].sum():>7,} voxels")
print(f"naive     Dice  {dice(c['truth'], masks_naive):.3f}   {masks_naive.sum():>7,} voxels")

### Claim 2 — backward propagation matters

The prompted slice sits mid-lesion, so a forward-only sweep should capture only part
of it.

In [ ]:
from medsam2_ct import init_state

c = cases[0]
state, _, _ = init_state(predictor, c["volume"])
predictor.add_new_points_or_box(
    inference_state=state, frame_idx=c["key"], obj_id=1,
    box=np.asarray(c["box"], dtype=np.float32))

forward = np.zeros_like(c["masks"])
for idx, _ids, logits in predictor.propagate_in_video(state):
    forward[idx] = (logits[0] > 0).cpu().numpy().squeeze()

print(f"bidirectional  Dice {c['dice']:.3f}   {c['masks'].sum():>7,} voxels")
print(f"forward only   Dice {dice(c['truth'], forward):.3f}   {forward.sum():>7,} voxels"
      f"   ({forward.sum() / c['masks'].sum():.0%})")

### Claim 3 — accuracy decays with distance from the prompt

Read this off the plot in section 4. Two things to look for: whether the extremes are
worse than the middle, and whether the prompted slice is actually the best one.

In [ ]:
for c in cases:
    z = c["truth"].any(axis=(1, 2)).nonzero()[0]
    per_slice = np.array([dice(c["truth"][i], c["masks"][i]) for i in z])
    offsets = z - c["key"]
    best = offsets[per_slice.argmax()]
    edges = per_slice[[0, -1]].mean()
    middle = per_slice[len(per_slice) // 4: -len(per_slice) // 4 or None].mean()
    print(f"{c['name']}  ({c['span']} slices)")
    print(f"   best slice at offset {best:+d}  (prompted slice is 0)")
    print(f"   mean Dice  middle {middle:.3f}   extremes {edges:.3f}")
    print()

---

Project: https://github.com/rekalantar/medsam2-3d-ct